In [43]:
import pandas as pd
import os
from pathlib import Path
import shutil
import re
import unicodedata

Cargar archivos csv

In [44]:
for folder in Path("../CSV ASIGNATURAS").iterdir():
    if folder.is_dir:
        asignatura = folder.name.split(" - ")[0].replace(" ", "_").lower()
        for file in folder.iterdir():
            tema = file.stem
            shutil.copy(file, Path("../csv_cleaned") / f"{asignatura}_{tema}.csv")

In [45]:
def create_id(row):
    id_string = f"{row["asignatura"]}_{row["año"]}_{row["convocatoria"]}"
    if not pd.isna(row["ejercicio"]):
        id_string += f"_{row["ejercicio"]}"
    if not pd.isna(row["opcion"]):
        id_string += f"_{row["opcion"]}"
    return id_string

In [46]:
HEADER = re.compile(r"""
    \b(?:F[IÍ]SICA|QU[IÍ]MICA|MATEM[AÁ]TICAS)   # asignatura (con o sin tilde)
    [A-ZÁÉÍÓÚÑ\s.]*?                            # resto del nombre: II, APLICADAS A LAS CC.SS. II...
    \.\s*(?:19|20)\d{2}\.                       # año
    (?:\s*\(?[A-ZÁÉÍÓÚÑ0-9]+\b\)?\.?)*          # convocatoria, ejercicio, opción...
""", re.VERBOSE)

def remove_exercise_info(texto):
    texto = HEADER.sub(" ", texto)
    return re.sub(r"\s+", " ", texto).strip()

In [47]:
def insert_new_line(statement):
    new_st = statement
    for point in ["a", "b", "c", "d"]:
        new_st = new_st.replace(f" {point})", f"\n{point})")
    return new_st

In [48]:
def clean_csv(file):
    df = pd.read_csv(file, encoding='utf-8')
    df_sin_metadatos = df[df.aviso == "sin metadatos"]
    df = df[df.aviso.isna()]

    # extract subject and topic
    asignatura = file.stem.split("_")[0]
    tema = "_".join(file.stem.split("_")[1:])
    df["asignatura"] = asignatura
    df["tema"] = tema

    # year to integer
    df.año = df.año.astype(int)

    # create id
    df["id"] = df.apply(lambda row: create_id(row), axis = 1)
    
    # rearrange columns
    df = df[["id"] + list(df.columns[:-2])]

    # remove year line from statement
    df["enunciado"] = df["enunciado"].apply(remove_exercise_info)

    # remove gravity from physics statements
    if asignatura == "física":
        df["enunciado"] = df.enunciado.str.replace("g = 9,8 m s-2 ", "")
        
    # insert new line in statement points
    df["enunciado"] = df.enunciado.apply(insert_new_line)

    # normalize text
    df.enunciado = df.enunciado.apply(lambda texto : unicodedata.normalize("NFKC", texto))

    # trim multiple spaces
    df.enunciado = df.enunciado.apply(lambda texto : re.sub(r"[ \t]+", " ", texto).strip())

    # export clean csv
    df.to_csv(file, encoding="utf-8", index = False)
    df_sin_metadatos.to_csv(f"{file}_sin_metadatos.csv", encoding="utf-8", index = False)

In [49]:
os.chdir("../csv_cleaned")

In [50]:
for file in Path(".").iterdir():
    if file.suffix == ".csv" and "sin_metadatos" not in file.name:
        try:
            clean_csv(file)
        except Exception as e:
            print(f"Error processing file: {file.name}")
            print(e)

In [36]:
df = pd.read_csv("química_redox.csv", encoding="utf-8")

In [37]:
df

,id,año,convocatoria,ejercicio,opcion,enunciado,imagen,aviso,asignatura
0,química_2000_Junio_6,2000,Junio,6,NaN,Dada la siguiente reacción redox en disolución...,enunciados_PNG/2000 - Reacciones Redox_png/200...,NaN,química
1,química_2000_Reserva 1_5_B,2000,Reserva 1,5,B,Dada la reacción redox en disolución acuosa: K...,enunciados_PNG/2000 - Reacciones Redox_png/200...,NaN,química
2,química_2000_Reserva 2_5_A,2000,Reserva 2,5,A,El ácido sulfúrico concentrado reacciona con e...,enunciados_PNG/2000 - Reacciones Redox_png/200...,NaN,química
3,química_2000_Reserva 3_3_A,2000,Reserva 3,3,A,a) ¿Reaccionará una disolución acuosa de ácido...,enunciados_PNG/2000 - Reacciones Redox_png/200...,NaN,química
4,química_2000_Reserva 4_3_A,2000,Reserva 4,3,A,De las siguientes reacciones: HCO  H CO H...,enunciados_PNG/2000 - Reacciones Redox_png/200...,NaN,química
...,...,...,...,...,...,...,...,...,...
205,química_2025_Reserva 1_3,2025,Reserva 1,3,NaN,Una forma de obtener C l 2 gaseoso en el labor...,enunciados_PNG/2025 - Reacciones Redox_png/202...,NaN,química
206,química_2025_Reserva 2_2,2025,Reserva 2,2,NaN,Dados los siguientes potenciales normales de r...,enunciados_PNG/2025 - Reacciones Redox_png/202...,NaN,química
207,química_2025_Reserva 2_3,2025,Reserva 2,3,NaN,El magnesio se obtiene por electrólisis de M g...,enunciados_PNG/2025 - Reacciones Redox_png/202...,NaN,química
208,química_2025_Julio_3,2025,Julio,3,NaN,Al hacer reaccionar ácido clorhídrico (HCl) co...,enunciados_PNG/2025 - Reacciones Redox_png/202...,NaN,química


In [38]:
df.isna().sum()

id                0
año               0
convocatoria      0
ejercicio         0
opcion           65
enunciado         1
imagen            0
aviso           210
asignatura        0
dtype: int64

In [39]:
df[df.enunciado.isna()]

,id,año,convocatoria,ejercicio,opcion,enunciado,imagen,aviso,asignatura
209,química_2026_Junio_5,2026,Junio,5,NaN,NaN,enunciados_PNG/2026 - Reacciones Redox_png/202...,NaN,química


In [29]:
df[df.enunciado.str.startswith("g = ")]

,id,año,convocatoria,ejercicio,opcion,enunciado,imagen,aviso,asignatura
1,física_2016_Junio_3_B,2016,Junio,3,B,"g = 9,8 m s-2 Una espira circular de 2,5 cm de...",enunciados_PNG/2016 - Campo eléctrico y magnét...,NaN,física
2,física_2016_Reserva 2_1_B,2016,Reserva 2,1,B,"g = 9,8 m s-2\na) Fuerza magnética sobre una c...",enunciados_PNG/2016 - Campo eléctrico y magnét...,NaN,física
4,física_2016_Reserva 3_1_A,2016,Reserva 3,1,A,"g = 9,8 m s-2\na) Enuncie la ley de inducción ...",enunciados_PNG/2016 - Campo eléctrico y magnét...,NaN,física
5,física_2016_Reserva 3_3_B,2016,Reserva 3,3,B,"g = 9,8 m s-2 Un haz de electrones con energía...",enunciados_PNG/2016 - Campo eléctrico y magnét...,NaN,física
6,física_2016_Reserva 4_1_A,2016,Reserva 4,1,A,"g = 9,8 m s-2\na) Enuncie la ley de Lenz-Farad...",enunciados_PNG/2016 - Campo eléctrico y magnét...,NaN,física
...,...,...,...,...,...,...,...,...,...
108,física_2025_Reserva_1,2025,Reserva,1,NaN,"g = 9,8 m s-2 Por un conductor rectilíneo muy ...",enunciados_PNG/2025 - Campo eléctrico y magnét...,NaN,física
111,física_2025_Reserva_2,2025,Reserva,2,NaN,"g = 9,8 m s-2 Dos partículas cargadas, con el ...",enunciados_PNG/2025 - Campo eléctrico y magnét...,NaN,física
112,física_2025_Reserva 2_b1,2025,Reserva 2,b1,NaN,"g = 9,8 m s-2 Una espira cuadrada contenida en...",enunciados_PNG/2025 - Campo eléctrico y magnét...,NaN,física
114,física_2025_Reserva_3,2025,Reserva,3,NaN,"g = 9,8 m s-2 Un protón entra en un campo magn...",enunciados_PNG/2025 - Campo eléctrico y magnét...,NaN,física
